# Minimal SARSA: Player-Only Blackjack (Beginner)

## Learning Goals

In this exercise, you will:

- Understand the core concepts: **state**, **actions**, **rewards**, and **episodes**
- Implement **ε-greedy action selection** to balance exploration and exploitation
- Implement **tabular SARSA update** with a pre-initialized Q table
- Run a quick demo and inspect the learned greedy policy

All using only basic Python (lists, dicts, loops, and the random module)!

## Rules of the Toy Environment

We use a simplified version of Blackjack:

- **State (s)**: Only the player's current sum matters. The state is an integer.
- **Cards**: Each card is a random integer from 1 to 10 (inclusive), all equally likely.
- **Actions**: 
  - `0 = stick` (end the episode)
  - `1 = hit` (draw another card)
- **Rewards**:
  - **stick**: Episode ends immediately. Reward = 0.
  - **hit**: Draw a card and add it to your sum.
    - If new sum ≤ 21: Reward = card value (1-10).
    - If new sum > 21: **bust!** Episode ends with reward = -sum.
- **Start state**: The player starts with sum = 0.

The goal is to learn when to stick vs. hit to maximize cumulative reward. You earn points for each card, but busting is very costly!

## Environment Code (Given)

Run this cell as-is. You do not need to edit it.

In [1]:
import random

def draw_card():
    # Cards are 1..10, equally likely
    return random.randint(1, 10)

# Actions: 0 = stick, 1 = hit
def take_action(sum_so_far, action):
    """
    Returns: (new_sum, reward, game_over)
    - hit: add a random card 1..10.
           If not bust: reward = card value.
           If bust (>21): reward = -sum, game_over.
    - stick: end episode with reward 0.
    """
    if action == 1:  # hit
        card = draw_card()
        s = sum_so_far + card
        if s > 21:
            return s, -s, True
        return s, card, False
    else:            # stick
        s = sum_so_far
        return s, 0, True

## 🎯 Try the Game!

Before learning SARSA, play the game yourself to understand it!

In [ ]:
def play_yourself():
    s = 0
    episode_reward = 0
    game_over = False
    
    print("Current sum: ", s)
    while not game_over:
        a = int(input("Action (0=STICK, 1=HIT): "))
        s_next, r, game_over = take_action(s, a)
        episode_reward += r
        s = s_next
        print("Current sum: ", s)
    
    print(f"Episode Reward: {episode_reward:+.0f}")

play_yourself()

## Pre-Initialized Q-Table Helper

We pre-allocate the Q-table so you never need to use `dict.get` or check for missing keys.

- **States**: 0 to 21 (inclusive) → 22 states total
- **Actions**: 0 (stick) and 1 (hit) → 2 actions
- **Q**: A list of length 22, where `Q[s]` is a list `[Q(s,0), Q(s,1)]`

All Q-values start at 0.0.

In [2]:
NUM_STATES = 22      # states 0..21 inclusive
Q_INIT_VALUE = 0.0   # start all Q-values at 0

def make_q_table():
    # Q is a list of [Q(s,0), Q(s,1)] for each state s
    Q = []
    for _ in range(NUM_STATES):
        Q.append([Q_INIT_VALUE, Q_INIT_VALUE])
    return Q

## ✍️ Student TODOs

Complete the two functions below:

1. **`choose_action(Q, state, epsilon)`**: Implement ε-greedy action selection
2. **`train_sarsa(...)`**: Implement the tabular SARSA loop

Read the comments carefully and follow the hints!

In [3]:
def choose_action(Q, state, epsilon):
    """
    Epsilon-greedy over actions {0,1} using Q[state][action].
    - With probability epsilon: pick a random action (0 or 1).
    - Otherwise: pick the action with the larger Q-value.
      If Q-values tie, you can choose any tie-break (e.g., pick 0).
    """
    # TODO: implement epsilon-greedy
    # Hints:
    #   if random.random() < epsilon: return random.randint(0, 1)
    #   q_stick = Q[state][0]
    #   q_hit   = Q[state][1]
    #   if q_hit > q_stick: return 1
    #   else: return 0
    return 0  # placeholder


def train_sarsa(episodes=200000, alpha=0.1, gamma=1.0, epsilon=0.1):
    """
    Train a tabular SARSA agent and return Q (list of [Q(s,0), Q(s,1)]).
    SARSA updates Q using the action you actually take next.
    
    Use only:
      - s = 0  (start state)
      - a = choose_action(Q, s, epsilon)
      - s_next, r, game_over = take_action(s, a)
      - if game_over:
          target = r
      - else:
          a_next = choose_action(Q, s_next, epsilon)
          target = r + gamma * Q[s_next][a_next]
      - Q[s][a] = Q[s][a] + alpha * (target - Q[s][a])
      - s = s_next
      - a = a_next (only when not game_over)
    """
    Q = make_q_table()

    # TODO: implement the SARSA loop described above

    return Q

## Quick Demo Runner

Run this cell **after** you complete the TODOs above.

It trains a SARSA agent briefly and prints the greedy policy (best action) for each state.

In [6]:
def greedy_action(Q, s):
    # epsilon = 0.0 -> choose best action
    if Q[s][1] > Q[s][0]:
        return 1
    return 0

# Run a short training and show the learned greedy action per state
# (Run this AFTER you complete the TODOs.)
try:
    Q_demo = train_sarsa(episodes=150000, alpha=0.1, gamma=1.0, epsilon=0.1)
    print("Greedy policy after training:")
    for s in range(0, 22):
        a = greedy_action(Q_demo, s)
        print(f"s={s:2d} -> {'HIT' if a == 1 else 'STICK'}   Q[s][stick]={Q_demo[s][0]:.2f} Q[s][hit]={Q_demo[s][1]:.2f}")
except Exception as e:
    print("Run error (did you complete the TODOs?):", e)

Greedy policy after training:
s= 0 -> HIT   Q[s][stick]=0.00 Q[s][hit]=14.53
s= 1 -> HIT   Q[s][stick]=0.00 Q[s][hit]=13.80
s= 2 -> HIT   Q[s][stick]=0.00 Q[s][hit]=12.62
s= 3 -> HIT   Q[s][stick]=0.00 Q[s][hit]=10.90
s= 4 -> HIT   Q[s][stick]=0.00 Q[s][hit]=10.03
s= 5 -> HIT   Q[s][stick]=0.00 Q[s][hit]=9.52
s= 6 -> HIT   Q[s][stick]=0.00 Q[s][hit]=8.80
s= 7 -> HIT   Q[s][stick]=0.00 Q[s][hit]=7.62
s= 8 -> HIT   Q[s][stick]=0.00 Q[s][hit]=7.76
s= 9 -> HIT   Q[s][stick]=0.00 Q[s][hit]=7.14
s=10 -> HIT   Q[s][stick]=0.00 Q[s][hit]=6.87
s=11 -> HIT   Q[s][stick]=0.00 Q[s][hit]=4.81
s=12 -> STICK   Q[s][stick]=0.00 Q[s][hit]=-2.71
s=13 -> STICK   Q[s][stick]=0.00 Q[s][hit]=-0.04
s=14 -> STICK   Q[s][stick]=0.00 Q[s][hit]=-4.06
s=15 -> STICK   Q[s][stick]=0.00 Q[s][hit]=-12.99
s=16 -> STICK   Q[s][stick]=0.00 Q[s][hit]=-10.92
s=17 -> STICK   Q[s][stick]=0.00 Q[s][hit]=-9.72
s=18 -> STICK   Q[s][stick]=0.00 Q[s][hit]=-17.21
s=19 -> STICK   Q[s][stick]=0.00 Q[s][hit]=-24.40
s=20 -> STICK   Q

## (Optional) Guided Questions

After running the demo, think about:

1. **What sum `s` becomes your "stick threshold" after training?**  
   Look at the policy above. Around what state does the agent switch from HIT to STICK?

2. **Why is `gamma=1.0` sensible here?**  
   This is an episodic task. Does discounting future rewards make sense?

3. **What happens if you increase epsilon to 0.3?**  
   Try re-running with `epsilon=0.3`. Does the policy change? Why or why not?

4. **Try fewer episodes (e.g., 10,000). Does the policy look noisier?**  
   Experiment with `episodes=10000`. Are the Q-values more uncertain?

## 📦 Instructor Solution

**Do not run this cell until revealing the solution!**

Below are the completed functions for reference.

In [5]:
# ===== INSTRUCTOR SOLUTION (do not run until revealing) =====

def choose_action(Q, state, epsilon):
    if random.random() < epsilon:
        return random.randint(0, 1)
    q_stick = Q[state][0]
    q_hit   = Q[state][1]
    if q_hit > q_stick:
        return 1
    return 0  # tie or stick better

def train_sarsa(episodes=200000, alpha=0.1, gamma=1.0, epsilon=0.1):
    Q = make_q_table()

    for _ in range(episodes):
        s = 0  # start state
        a = choose_action(Q, s, epsilon)

        while True:
            s_next, r, game_over = take_action(s, a)
            
            # SARSA update
            if game_over:
                target = r
            else:
                a_next = choose_action(Q, s_next, epsilon)
                target = r + gamma * Q[s_next][a_next]
            
            Q[s][a] = Q[s][a] + alpha * (target - Q[s][a])

            if game_over:
                break
                
            s = s_next
            a = a_next

    return Q

## 🎮 Watch the Agent Play

See how the trained agent plays the game!

In [ ]:
def play_game(Q):
    s = 0
    episode_reward = 0
    game_over = False
    
    print("Current sum: ", s)
    while not game_over:
        a = greedy_action(Q, s)
        s_next, r, game_over = take_action(s, a)
        episode_reward += r
        s = s_next
        print("Current sum: ", s)
    
    print(f"Episode Reward: {episode_reward:+.0f}")
    return episode_reward

for i in range(3):
    print(f"Game {i+1}:")
    play_game(Q_demo)